In [2]:
import pandas as pd
import re

df_final = pd.read_csv('df_final.csv', encoding='utf-8-sig')
print(f"로드 완료: {len(df_final):,}건")
print(f"컬럼 목록: {df_final.columns.tolist()}")

C:\Users\hkhk3\AppData\Local\Temp\ipykernel_29032\734045890.py:4: DtypeWarning: Columns (0: 만족도, 1: 사이즈, 2: 화면대비색감, 3: 퀄리티, 4: 구김, 5: 두께감, 6: 신축성, 7: 색감, 8: 보온성) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv('df_final.csv', encoding='utf-8-sig')


로드 완료: 623,378건
컬럼 목록: ['리뷰번호', 'goodsNo', '리뷰타입', '작성자', '리뷰내용', '평점', '체험단', '구매옵션', '키', '몸무게', '성별', '작성일', '만족도', '사진유무', '도움돼요', '구매사이즈', '구매상세', '연도', '월', '일', '요일', '평점_raw', '만족도_응답여부', '사이즈', '화면대비색감', '퀄리티', '구김', '두께감', '신축성', '색감', '보온성', '퀄리티_점수', '보온성_점수', '신축성_점수', '두께감_점수', '구김_점수', '사이즈_편차', '사이즈_편차절대', '화면대비색감_편차', '화면대비색감_편차절대', '색감_편차', '색감_편차절대', '노출일수', '일평균_도움돼요지수', '도움여부', '리뷰내용_clean', '리뷰내용_norm', 'laugh_count', 'cry_count', 'exclamation_count', 'question_count', 'emoji_count', 'has_repetition', 'text_length_orig', '한글_글자수', '전체_글자수', 'is_valid_for_topic', '작성자_norm', '옵션키', '세션', 'long_gap_review', '그룹크기', 'action', 'dup_flag', 'is_base', 'kept_id']


In [5]:
from kiwipiepy import Kiwi
from collections import Counter

kiwi = Kiwi()

all_nouns = []
for text in df_final['리뷰내용_clean'].dropna():
    for token in kiwi.tokenize(str(text)):
        word = token.form
        pos  = token.tag
        if pos in {'NNG', 'NNP', 'SL'}:
            if len(word) > 1:
                all_nouns.append(word)

counter = Counter(all_nouns)
print(counter.most_common(500))

[('사이즈', 115724), ('생각', 69066), ('구매', 59617), ('만족', 54583), ('가격', 50988), ('오버핏', 49203), ('색감', 48069), ('재질', 41874), ('가성비', 35023), ('마음', 34125), ('여름', 33283), ('느낌', 32954), ('배송', 32344), ('바지', 31821), ('두께', 30190), ('기장', 28719), ('디자인', 28368), ('색상', 28281), ('후드', 26766), ('추천', 25905), ('기모', 23937), ('길이', 20604), ('정도', 18287), ('겨울', 18192), ('허리', 16100), ('색깔', 15492), ('부분', 15199), ('제품', 14490), ('대비', 14371), ('레이어드', 14309), ('사진', 13790), ('가을', 13263), ('기본', 13066), ('이너', 11859), ('원단', 11265), ('최고', 11198), ('스타일', 10535), ('오버', 10502), ('세탁', 10308), ('주문', 10172), ('소재', 10053), ('요즘', 9483), ('감사', 9313), ('날씨', 9248), ('퀄리티', 9002), ('코디', 8767), ('고민', 8720), ('모자', 8541), ('처음', 8384), ('셔츠', 8235), ('친구', 8032), ('상품', 7913), ('어깨', 7888), ('데일리', 7405), ('구입', 7374), ('반팔', 7324), ('맨투맨', 7320), ('프린팅', 7139), ('블랙', 6657), ('포인트', 6492), ('전체', 6319), ('시보리', 6188), ('다음', 6155), ('마감', 6121), ('세일', 6101), ('그레이', 6100), ('자체', 5894), ('선물'

In [ ]:
from kiwipiepy import Kiwi

kiwi = Kiwi()

# 불용어 정의 (BERTopic 토픽 키워드 추출용)
custom_stopwords = {
    # 쇼핑 관용어
    "구매", "구입", "주문", "제품", "상품", "추천", "만족",
    "감사", "후기", "사진", "착용", "사용", "교환",
    "환불", "반품", "할인", "세일", "쿠폰", "예약", "품절", "리뷰",
    # 너무 일반적인 단어
    "생각", "마음", "느낌", "정도", "부분", "기본", "처음", "다음",
    "이번", "필요", "전체", "자체", "사람", "친구", "경우", "상태",
    "문제", "이상", "기간", "시간", "기준", "활용", "선택", "이유",
    "확인", "참고", "예상", "나중", "작년", "올해", "하루", "동안",
    "지금", "원래", "기존", "실제", "보통", "중간",
    # 의미 약한 단어
    "요즘", "최고", "고민", "걱정", "기대", "선물", "화면", "평소",
    "포인트", "완전", "대신", "나머지", "약간", "진짜", "덕분",
    "다행", "당황", "무리", "조심", "제외", "비교", "조합", "활용도",
    # 가족/지인
    "동생", "아들", "남편", "아빠", "엄마", "오빠",
    "남친", "와이프", "남동생", "신랑", "여성분",
    # 오타/구어체
    "조아", "조아요", "싸이즈", "두깨", "두깨감",
}

# 한국어 리뷰 문장을 토큰(단어) 리스트로 변환하는 함수
def tokenize_ko(text):
    tokens = []

    # kiwi.tokenize(text)
    # 반환 예시:
    # [Token(form='배송', tag='NNG', ...), Token(form='늦다', tag='VA', ...)]
    for token in kiwi.tokenize(str(text)):
        word = token.form
        pos  = token.tag

        # 명사만 사용 (BERTopic 토픽 키워드는 명사 기반)
        # NNG: 일반명사, NNP: 고유명사, SL: 영문
        if pos in {'NNG', 'NNP', 'SL'}:
            # 길이 1인 토큰 제거 + 불용어 제거
            if len(word) > 1 and word not in custom_stopwords:
                tokens.append(word)

    return tokens

print(kiwi.tokenize("배송이 너무 늦고 포장이 부실해서 실망했어요"))
print("포장" in custom_stopwords)

[Token(form='배송', tag='NNG', start=0, len=2), Token(form='이', tag='JKS', start=2, len=1), Token(form='너무', tag='MAG', start=4, len=2), Token(form='늦', tag='VA', start=7, len=1), Token(form='고', tag='EC', start=8, len=1), Token(form='포장', tag='NNG', start=10, len=2), Token(form='이', tag='JKS', start=12, len=1), Token(form='부실', tag='NNG', start=14, len=2), Token(form='하', tag='XSA', start=16, len=1), Token(form='어서', tag='EC', start=16, len=2), Token(form='실망', tag='NNG', start=19, len=2), Token(form='하', tag='XSV', start=21, len=1), Token(form='었', tag='EP', start=21, len=1), Token(form='어요', tag='EF', start=22, len=2)]
True
